# Example 03: Cross-Split Sample Duplication Leakage (Type 2)

**Planted Flaw**: Premature duplication / naive oversampling is applied to the full dataset before splitting, causing identical feature rows to appear in both train and test splits.
The labels contain realistic noise (`flip_y=0.2`), so memorised duplicates inflate the score clearly.
**Ground Truth**: `CONFIRMED` Type 2 Leakage.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
# Generate base classification dataset
np.random.seed(42)
X_raw, y_raw = make_classification(
    n_samples=200,
    n_features=20,
    n_informative=10,
    flip_y=0.2,
    random_state=42
)

# FLAGGED LEAK: Naive duplication / oversampling applied before split
dup_indices = np.random.choice(len(X_raw), size=100, replace=True)
X = np.vstack([X_raw, X_raw[dup_indices]])
y = np.concatenate([y_raw, y_raw[dup_indices]])

In [ ]:
# Naive random split places duplicate rows in BOTH train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [ ]:
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_prob)
print(f"Duplicated Split Test Accuracy: {test_acc:.4f}")
print(f"Duplicated Split Test ROC-AUC: {test_auc:.4f}")